In [ ]:
import torch
from torchvision import datasets
import matplotlib.pyplot as plt
import numpy as np

# Tải bộ dữ liệu FashionMNIST
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)

# Lưu trữ hình ảnh và nhãn (targets)
tr_images = fmnist.data
tr_targets = fmnist.targets

In [ ]:
unique_values = tr_labels.unique()
print(f'tr_image & tr_targets: \n\tX - {tr_images.shape} \n\ty - {tr_labels.shape} \n\tY - unique values : {unique_values}')
print(f'TASK:\n\t{len(unique_values)} class classification')
print(f'UNIQUE CLASSES:\n\t{fmnist.classes}')

In [ ]:
R, C = len(tr_targets.unique()), 10
fig, ax = plt.subplots(R, C, figsize=(10, 10))

for label_class, plot_row in enumerate(ax):
    # Lấy chỉ mục của các ảnh thuộc lớp hiện tại
    label_x_rows = np.where(tr_targets == label_class)[0]
    
    for plot_cell in plot_row:
        plot_cell.grid(False)
        plot_cell.axis('off')
        
        # Chọn ngẫu nhiên ảnh và hiển thị
        ix = np.random.choice(label_x_rows)
        x = tr_images[ix]
        plot_cell.imshow(x, cmap='gray')

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from torchvision import datasets
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD # Sử dụng lại SGD thay vì Adam

# Thiết lập cấu hình phần cứng tăng tốc
device = "cuda" if torch.cuda.is_available() else "cpu"

# Tải dữ liệu FMNIST cho cả tập Train và Validation
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
tr_images = fmnist.data
tr_targets = fmnist.targets

val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)
val_images = val_fmnist.data
val_targets = val_fmnist.targets

# Lớp Dataset tùy biến tích hợp chuẩn hóa /255
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float() / 255.0
        x = x.view(-1, 28 * 28)
        self.x, self.y = x, y
        
    def __getitem__(self, ix):
        x, y = self.x[ix], self.y[ix]
        return x.to(device), y.to(device)
        
    def __len__(self):
        return len(self.x)

def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from torch.utils.data import DataLoader
from torch.optim import Adam

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- KHỞI TẠO LẠI BATCH SIZE LỚN (10,000) ---
def get_data_large_batch():
    train = FMNISTDataset(tr_images, tr_targets)
    # Cấu hình batch_size = 10000
    trn_dl = DataLoader(train, batch_size=10000, shuffle=True)
    
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

trn_dl, val_dl = get_data_large_batch()
model, loss_fn, optimizer = get_model() # Sử dụng cấu hình model tối ưu trước đó

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

# Tiến hành huấn luyện kéo dài đủ 10 Kỷ nguyên (Epochs) theo tài liệu mới
for epoch in range(10):
    print(f"Epoch đang chạy (Batch Size 10k - 10 Epochs): {epoch}")
    train_epoch_losses, train_epoch_accuracies = [], []
    
    # 1. Huấn luyện cập nhật trọng số trên tập Train
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
    train_epoch_loss = np.array(train_epoch_losses).mean()
    
    # 2. Tính độ chính xác tập Train
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    train_epoch_accuracy = np.mean(train_epoch_accuracies)
    
    # 3. Đánh giá trên tập dữ liệu Xác thực (Validation Set)
    for ix, batch in enumerate(iter(val_dl)):
        x, y = batch
        val_loss_value = val_loss(x, y, model, loss_fn)
        val_is_correct = accuracy(x, y, model)
    val_epoch_accuracy = np.mean(val_is_correct)
    
    # Lưu trữ kết quả lịch sử
    train_losses.append(train_epoch_loss)
    train_accuracies.append(train_epoch_accuracy)
    val_losses.append(val_loss_value)
    val_accuracies.append(val_epoch_accuracy)

In [ ]:
# Cấu hình đồ thị hiển thị kết quả qua 10 Epochs
epochs = np.arange(10) + 1
plt.figure(figsize=(10, 10))

# Biểu đồ Loss (Training vs Validation với Batch Size = 10000)
plt.subplot(211)
plt.plot(epochs, train_losses, 'bo', label='Training loss')
plt.plot(epochs, val_losses, 'r', label='Validation loss')
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation loss when batch size is 10000')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(False)

# Biểu đồ Accuracy (Training vs Validation với Batch Size = 10000)
plt.subplot(212)
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r', label='Validation accuracy')
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation accuracy when batch size is 10000')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x * 100) for x in plt.gca().get_yticks()])
plt.legend()
plt.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from torch.optim import SGD

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- THAY ĐỔI SIÊU THAM SỐ: SGD VỚI LEARNING RATE = 0.1 ---
def get_model_sgd_high_lr():
    model = nn.Sequential(
        nn.Linear(28 * 28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    
    loss_fn = nn.CrossEntropyLoss()
    # Sử dụng bộ tối ưu SGD với lr = 0.1 (Mức học suất lớn đối với SGD)
    optimizer = SGD(model.parameters(), lr=1e-1)
    return model, loss_fn, optimizer

In [ ]:
# Gọi hàm nạp dữ liệu từ các bước trước
trn_dl, val_dl = get_data()

model, loss_fn, optimizer = get_model_sgd_high_lr()
train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(5):
    print(f"Epoch đang chạy (SGD với LR = 0.1): {epoch}")
    train_epoch_losses, train_epoch_accuracies = [], []
    
    # 1. Huấn luyện cập nhật trọng số trên tập Train
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
    train_epoch_loss = np.array(train_epoch_losses).mean()
    
    # 2. Tính độ chính xác tập Train
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    train_epoch_accuracy = np.mean(train_epoch_accuracies)
    
    # 3. Đánh giá trên tập Validation
    for ix, batch in enumerate(iter(val_dl)):
        x, y = batch
        val_loss_value = val_loss(x, y, model, loss_fn)
        val_is_correct = accuracy(x, y, model)
    val_epoch_accuracy = np.mean(val_is_correct)
    
    # Lưu lịch sử kết quả qua từng Epoch
    train_losses.append(train_epoch_loss)
    train_accuracies.append(train_epoch_accuracy)
    val_losses.append(val_loss_value)
    val_accuracies.append(val_epoch_accuracy)

In [ ]:
epochs = np.arange(5) + 1
plt.figure(figsize=(10, 10))

# Biểu đồ Loss (Training vs Validation với SGD, LR = 0.1)
plt.subplot(211)
plt.plot(epochs, train_losses, 'bo', label='Training loss')
plt.plot(epochs, val_losses, 'r', label='Validation loss')
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation loss with SGD and 0.1 learning rate')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(False)

# Biểu đồ Accuracy (Training vs Validation với SGD, LR = 0.1)
plt.subplot(212)
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r', label='Validation accuracy')
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation accuracy with SGD and 0.1 learning rate')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x * 100) for x in 
plt.gca().get_yticks()])
plt.legend()
plt.grid('off')

plt.tight_layout()
plt.show()